<a href="https://colab.research.google.com/github/yejinPARK48/michigan_building_detection/blob/main/04_threshold_tuning/washtenaw_reeval_threshold_tuned_model_phase4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Washtenaw County — Re-evaluation with Phase 4 Best Config

This notebook re-runs inference on the Washtenaw County tiles using the
Phase 4 best post-processing configuration (**threshold=0.7, min_area=50**)
and compares tile-level and block-group-level MAE / Pearson r against the
previous run (threshold=0.5, min_area=50).

No re-download of TIGER shapefiles or Microsoft Building Footprints is needed —
ground-truth `footprint_count` and `tier` are already stored in
`washtenaw_tile_metadata.csv` from the previous run.


## 0. Setup

In [ ]:
# ── SSL patch MUST be first before any other imports ─────────────────────
import os, ssl, urllib3

os.environ['CURL_CA_BUNDLE']     = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings()


In [ ]:
!pip install pystac-client planetary-computer rasterio segmentation-models-pytorch albumentations opencv-python-headless scikit-image -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 14.8 MB/s eta 0:00:00


In [ ]:
import time
import numpy as np
import pandas as pd
from PIL import Image
from skimage import measure

import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.crs import CRS
from rasterio.warp import transform_bounds

import pystac_client
import planetary_computer as pc
pc.settings.set_subscription_key("0162e7782ab28b85cda3c77b61875ca3210a1c31")

import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from google.colab import drive
drive.mount('/content/drive')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


Mounted at /content/drive
Device: cuda


## 0-2. Config — Phase 4 Best Configuration

In [ ]:
BASE_DIR      = '/content/drive/MyDrive/michigan_unet_project'
CKPT_DIR      = f'{BASE_DIR}/checkpoints'
SEG_CKPT      = f'{CKPT_DIR}/unet_seg_best_full_curated.pt'

OUT_DIR       = f'{BASE_DIR}/results_washtenaw'
TILE_IMG_DIR  = '/content/washtenaw/images'   # may already be populated from the previous run

# ── New config from Phase 4 sweep ──────────────────────────────────────
BEST_THRESHOLD = 0.7
BEST_MIN_AREA  = 50

# ── Previous run config, for comparison ──────────────────────────────────
OLD_THRESHOLD  = 0.5
OLD_MIN_AREA   = 50

DENSE_THRESHOLD  = 0.05
SPARSE_THRESHOLD = 0.001

def classify_tier(ratio):
    if ratio >= DENSE_THRESHOLD:    return 'Dense'
    elif ratio >= SPARSE_THRESHOLD: return 'Sparse'
    else:                           return 'Empty'

os.makedirs(TILE_IMG_DIR, exist_ok=True)
print(f'Checkpoint exists : {os.path.exists(SEG_CKPT)}')
print(f'New config  : threshold={BEST_THRESHOLD}, min_area={BEST_MIN_AREA}')
print(f'Old config  : threshold={OLD_THRESHOLD}, min_area={OLD_MIN_AREA}')


Checkpoint exists : True
New config  : threshold=0.7, min_area=50
Old config  : threshold=0.5, min_area=50


## 1. Load Existing Tile Metadata

Ground-truth `footprint_count`, `tier`, and `bbox` columns are reused from the previous Washtenaw run — no need to re-download MS Footprints or TIGER shapefiles.

In [ ]:
META_CSV = f'{OUT_DIR}/washtenaw_tile_metadata.csv'
df_tiles = pd.read_csv(META_CSV, dtype={'geoid': str, 'tile_id': str})
print(f'Loaded {len(df_tiles)} tiles')
print(df_tiles[['tile_id','geoid','footprint_count','mask_building_ratio','tier']].head())


Loaded 32703 tiles
                      tile_id         geoid  footprint_count  \
0  washtenaw_261614008002_000  261614008002              100   
1  washtenaw_261614101001_000  261614101001               16   
2  washtenaw_261614054001_000  261614054001               52   
3  washtenaw_261614054001_016  261614054001               15   
4  washtenaw_261614054001_017  261614054001               59   

   mask_building_ratio   tier  
0             0.224998  Dense  
1             0.071697  Dense  
2             0.104706  Dense  
3             0.162796  Dense  
4             0.128723  Dense  


## 2. Ensure Tile Images Are Available

If the runtime was restarted, local images under `/content/washtenaw/images` will be gone. Any missing tiles are re-fetched from NAIP using the stored `bbox` columns (no mask regeneration needed).

In [ ]:
import concurrent.futures
from threading import Lock

def fetch_naip_tile(bbox, year=2020, size_px=512, max_retries=3):
    for attempt in range(max_retries):
        try:
            catalog = pystac_client.Client.open(
                'https://planetarycomputer.microsoft.com/api/stac/v1',
                modifier=pc.sign_inplace
            )
            items = catalog.search(
                collections=['naip'], bbox=bbox,
                datetime=f'{year}-01-01/{year}-12-31', max_items=5
            ).item_collection()
            if len(items) == 0: return None
            href = pc.sign(items[0].assets['image'].href)
            with rasterio.open(href) as src:
                bbox_transformed = transform_bounds(
                    CRS.from_epsg(4326), src.crs,
                    bbox[0], bbox[1], bbox[2], bbox[3]
                )
                window = from_bounds(*bbox_transformed, transform=src.transform)
                data   = src.read([1,2,3], window=window,
                                   out_shape=(3, size_px, size_px),
                                   resampling=Resampling.bilinear)
            return np.transpose(data, (1,2,0)).astype(np.uint8)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                return None
    return None


missing = [t for t in df_tiles['tile_id'] if not os.path.exists(f'{TILE_IMG_DIR}/{t}.png')]
print(f'Missing images: {len(missing)} / {len(df_tiles)}')

if missing:
    print('Re-fetching missing NAIP tiles in parallel...')
    bbox_map = df_tiles.set_index('tile_id')[['bbox_minx','bbox_miny','bbox_maxx','bbox_maxy']]

    lock = Lock()
    fetched, failed = [0], [0]

    def fetch_and_save(tid):
        bbox = tuple(bbox_map.loc[tid])
        img_np = fetch_naip_tile(bbox)
        if img_np is None:
            with lock:
                failed[0] += 1
            return
        Image.fromarray(img_np).save(f'{TILE_IMG_DIR}/{tid}.png')
        with lock:
            fetched[0] += 1

    WORKERS = 4  # lower to 4 if rate-limited

    with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = [executor.submit(fetch_and_save, tid) for tid in missing]
        for i, _ in enumerate(concurrent.futures.as_completed(futures)):
            if (i + 1) % 500 == 0:
                print(f'  progress={i+1}/{len(missing)} fetched={fetched[0]} failed={failed[0]}')

    print(f'Done. Fetched={fetched[0]}, Failed={failed[0]}')
else:
    print('All tile images already cached locally.')

Missing images: 32703 / 32703
Re-fetching missing NAIP tiles in parallel...
  progress=500/32703 fetched=500 failed=0
  progress=1000/32703 fetched=1000 failed=0
  progress=1500/32703 fetched=1500 failed=0
  progress=2000/32703 fetched=2000 failed=0
  progress=2500/32703 fetched=2500 failed=0
  progress=3000/32703 fetched=3000 failed=0
  progress=3500/32703 fetched=3500 failed=0
  progress=4000/32703 fetched=4000 failed=0
  progress=4500/32703 fetched=4500 failed=0
  progress=5000/32703 fetched=5000 failed=0
  progress=5500/32703 fetched=5500 failed=0
  progress=6000/32703 fetched=6000 failed=0
  progress=6500/32703 fetched=6500 failed=0
  progress=7000/32703 fetched=7000 failed=0
  progress=7500/32703 fetched=7500 failed=0
  progress=8000/32703 fetched=8000 failed=0
  progress=8500/32703 fetched=8500 failed=0
  progress=9000/32703 fetched=9000 failed=0
  progress=9500/32703 fetched=9500 failed=0
  progress=10000/32703 fetched=10000 failed=0
  progress=10500/32703 fetched=10500 failed=

  progress=24500/32703 fetched=24500 failed=0


  progress=25000/32703 fetched=25000 failed=0
  progress=25500/32703 fetched=25500 failed=0
  progress=26000/32703 fetched=26000 failed=0
  progress=26500/32703 fetched=26500 failed=0
  progress=27000/32703 fetched=27000 failed=0
  progress=27500/32703 fetched=27500 failed=0
  progress=28000/32703 fetched=28000 failed=0
  progress=28500/32703 fetched=28500 failed=0
  progress=29000/32703 fetched=29000 failed=0
  progress=29500/32703 fetched=29500 failed=0
  progress=30000/32703 fetched=30000 failed=0
  progress=30500/32703 fetched=30500 failed=0
  progress=31000/32703 fetched=31000 failed=0
  progress=31500/32703 fetched=31500 failed=0
  progress=32000/32703 fetched=32000 failed=0
  progress=32500/32703 fetched=32500 failed=0
Done. Fetched=32703, Failed=0


## 3. Model Inference with Phase 4 Best Config

In [ ]:
model = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                 in_channels=3, classes=1, activation=None).to(DEVICE)
model.load_state_dict(torch.load(SEG_CKPT, map_location=DEVICE))
model.eval()
print('Model loaded.')

val_aug = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class InferenceDataset(Dataset):
    def __init__(self, tile_ids, img_dir, transform=None):
        self.tile_ids  = tile_ids
        self.img_dir   = img_dir
        self.transform = transform
    def __len__(self): return len(self.tile_ids)
    def __getitem__(self, idx):
        tid = self.tile_ids[idx]
        img = np.array(Image.open(f'{self.img_dir}/{tid}.png').convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, tid

# Only run on tiles whose images are actually available
available_ids = [t for t in df_tiles['tile_id'] if os.path.exists(f'{TILE_IMG_DIR}/{t}.png')]
print(f'Running inference on {len(available_ids)} / {len(df_tiles)} tiles')

inf_ds = InferenceDataset(available_ids, TILE_IMG_DIR, val_aug)
inf_ld = DataLoader(inf_ds, batch_size=8, shuffle=False, num_workers=2)

gt_map = dict(zip(df_tiles['tile_id'], df_tiles['footprint_count']))

results_v2 = []
with torch.no_grad():
    for imgs, tids in inf_ld:
        seg_logits = model(imgs.to(DEVICE))
        pred_masks = (torch.sigmoid(seg_logits) > BEST_THRESHOLD).cpu().numpy()
        for b in range(len(tids)):
            tid     = tids[b]
            labeled = measure.label(pred_masks[b, 0])
            cc_cnt  = sum(1 for r in measure.regionprops(labeled) if r.area >= BEST_MIN_AREA)
            results_v2.append({'tile_id': tid, 'cc_count': cc_cnt,
                                'gt_count': gt_map.get(tid, 0)})

df_results_v2 = pd.DataFrame(results_v2)
df_results_v2 = df_results_v2.merge(
    df_tiles[['tile_id','geoid','mask_building_ratio','tier']],
    on='tile_id')

OUT_CSV_V2 = f'{OUT_DIR}/washtenaw_predictions_thr{BEST_THRESHOLD}_minarea{BEST_MIN_AREA}.csv'
df_results_v2.to_csv(OUT_CSV_V2, index=False)
print(f'Saved: {OUT_CSV_V2}')

mae_v2  = np.abs(df_results_v2['cc_count'] - df_results_v2['gt_count']).mean()
corr_v2 = np.corrcoef(df_results_v2['cc_count'], df_results_v2['gt_count'])[0,1]
print(f'\n[New config: thr={BEST_THRESHOLD}, min_area={BEST_MIN_AREA}]')
print(f'Tile-level MAE: {mae_v2:.2f} | Pearson r: {corr_v2:.4f}')


Model loaded.
Running inference on 32703 / 32703 tiles
Saved: /content/drive/MyDrive/michigan_unet_project/results_washtenaw/washtenaw_predictions_thr0.7_minarea50.csv

[New config: thr=0.7, min_area=50]
Tile-level MAE: 1.40 | Pearson r: 0.9553


## 4. Block Group Aggregation

In [ ]:
bg_results_v2 = df_results_v2.groupby('geoid').agg(
    pred_cc_count = ('cc_count',  'sum'),
    gt_fp_count   = ('gt_count',  'sum'),
    n_tiles       = ('tile_id',   'count'),
    mean_ratio    = ('mask_building_ratio', 'mean'),
).reset_index()

bg_results_v2['bg_tier'] = bg_results_v2['mean_ratio'].apply(classify_tier)

mae_bg_v2  = np.abs(bg_results_v2['pred_cc_count'] - bg_results_v2['gt_fp_count']).mean()
corr_bg_v2 = np.corrcoef(bg_results_v2['pred_cc_count'], bg_results_v2['gt_fp_count'])[0,1]

OUT_BG_CSV_V2 = f'{OUT_DIR}/washtenaw_bg_predictions_thr{BEST_THRESHOLD}_minarea{BEST_MIN_AREA}.csv'
bg_results_v2.to_csv(OUT_BG_CSV_V2, index=False)
print(f'Saved: {OUT_BG_CSV_V2}')

print(f'\n[New config: thr={BEST_THRESHOLD}, min_area={BEST_MIN_AREA}]')
print(f'Block groups   : {len(bg_results_v2)}')
print(f'Block-group MAE: {mae_bg_v2:.2f} | Pearson r: {corr_bg_v2:.4f}')


Saved: /content/drive/MyDrive/michigan_unet_project/results_washtenaw/washtenaw_bg_predictions_thr0.7_minarea50.csv

[New config: thr=0.7, min_area=50]
Block groups   : 308
Block-group MAE: 89.63 | Pearson r: 0.9648


## 5. Tier-Stratified Evaluation (Tile Level)

In [ ]:
print(f'[New config: thr={BEST_THRESHOLD}, min_area={BEST_MIN_AREA}]')
print(f'{"Tier":<8} | {"N":>5} | {"MAE":>8} | {"Pearson r":>10}')
print('-' * 40)

tier_results_v2 = {}
for tier in ['Dense', 'Sparse', 'Empty']:
    sub = df_results_v2[df_results_v2['tier'] == tier]
    if len(sub) == 0: continue
    mae_t  = np.abs(sub['cc_count'] - sub['gt_count']).mean()
    corr_t = np.corrcoef(sub['cc_count'], sub['gt_count'])[0,1] if sub['cc_count'].std() > 0 else 0.0
    tier_results_v2[tier] = {'n': len(sub), 'mae': mae_t, 'r': corr_t}
    print(f'{tier:<8} | {len(sub):>5} | {mae_t:>8.2f} | {corr_t:>10.4f}')


[New config: thr=0.7, min_area=50]
Tier     |     N |      MAE |  Pearson r
----------------------------------------
Dense    |  4022 |     6.13 |     0.9177
Sparse   | 14286 |     1.34 |     0.8390
Empty    | 14395 |     0.15 |     0.1855


## 6. Comparison with Previous Run (threshold=0.5)

In [ ]:
OLD_CSV    = f'{OUT_DIR}/washtenaw_predictions.csv'
OLD_BG_CSV = f'{OUT_DIR}/washtenaw_bg_predictions.csv'

if os.path.exists(OLD_CSV) and os.path.exists(OLD_BG_CSV):
    df_old    = pd.read_csv(OLD_CSV,    dtype={'geoid': str})
    bg_old    = pd.read_csv(OLD_BG_CSV, dtype={'geoid': str})

    mae_old     = np.abs(df_old['cc_count'] - df_old['gt_count']).mean()
    corr_old    = np.corrcoef(df_old['cc_count'], df_old['gt_count'])[0,1]
    mae_bg_old  = np.abs(bg_old['pred_cc_count'] - bg_old['gt_fp_count']).mean()
    corr_bg_old = np.corrcoef(bg_old['pred_cc_count'], bg_old['gt_fp_count'])[0,1]

    print('=' * 60)
    print(f'{"Metric":<28} | {"Old (thr=0.5)":>14} | {"New (thr=0.7)":>14}')
    print('-' * 60)
    print(f'{"Tile-level MAE":<28} | {mae_old:>14.2f} | {mae_v2:>14.2f}')
    print(f'{"Tile-level Pearson r":<28} | {corr_old:>14.4f} | {corr_v2:>14.4f}')
    print(f'{"Block group MAE":<28} | {mae_bg_old:>14.2f} | {mae_bg_v2:>14.2f}')
    print(f'{"Block group Pearson r":<28} | {corr_bg_old:>14.4f} | {corr_bg_v2:>14.4f}')
    print('-' * 60)

    print(f'\n{"Tier":<8} | {"MAE (old)":>10} | {"MAE (new)":>10} | {"r (old)":>9} | {"r (new)":>9}')
    for tier in ['Dense', 'Sparse', 'Empty']:
        sub_old = df_old[df_old['tier'] == tier]
        sub_new = df_results_v2[df_results_v2['tier'] == tier]
        mae_o = np.abs(sub_old['cc_count'] - sub_old['gt_count']).mean()
        mae_n = tier_results_v2[tier]['mae']
        r_o   = np.corrcoef(sub_old['cc_count'], sub_old['gt_count'])[0,1] if sub_old['cc_count'].std() > 0 else 0.0
        r_n   = tier_results_v2[tier]['r']
        print(f'{tier:<8} | {mae_o:>10.2f} | {mae_n:>10.2f} | {r_o:>9.4f} | {r_n:>9.4f}')
else:
    print('Previous results CSVs not found — skipping comparison.')


Metric                       |  Old (thr=0.5) |  New (thr=0.7)
------------------------------------------------------------
Tile-level MAE               |           1.42 |           1.40
Tile-level Pearson r         |         0.9543 |         0.9553
Block group MAE              |          90.13 |          89.63
Block group Pearson r        |         0.9636 |         0.9648
------------------------------------------------------------

Tier     |  MAE (old) |  MAE (new) |   r (old) |   r (new)
Dense    |       6.26 |       6.13 |    0.9155 |    0.9177
Sparse   |       1.33 |       1.34 |    0.8419 |    0.8390
Empty    |       0.16 |       0.15 |    0.1896 |    0.1855
